# Note:
- The data from the [Chicago Data Portal](https://data.cityofchicago.org/browse?category=Public+Safety&sortBy=most_accessed&page=1&pageSize=20) and Crime Data set were enriched using multiple datasets from the portal. We initially stored them in the PostgreSQL database to generate the enriched dataset by joining multiple datasets using the_geom, but we discovered inconsistencies in the police beat, district, and sector fields. All missing fields were determined using multiple fields to generate the most accurate information, but there may be errors during the data wrangling process.

- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation using data from other sources to fill corresponding NaN entries in location-based fields.

In [1]:
# import libraries
from platform import python_version
import sys
import time
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import re

# python source path
sys.path.append('../Src/')

# random seed
_RANDOM_STATE = 1776
    
# python
import utils
import geo
import geo_dict

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# Use a single Arrow string & int64 type instance to save memory
arrow_string = pd.ArrowDtype(pa.string())
arrow_float64 = pd.ArrowDtype(pa.float64())
arrow_int64 = pd.ArrowDtype(pa.int64())
arrow_int8 = pd.ArrowDtype(pa.int8())

# for low cardinality categorical columns (2–128 unique values)
arrow_cat8  = pd.ArrowDtype(pa.dictionary(
                  index_type=pa.int8(), 
                  value_type=pa.string()
              ))

# capture time
start = time.time()

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0


## Read Data
- The Chicago Crime Data contains Crime, Arrest, IUCR, Neighborhood, and Police Beat datasets from the Chicago Crime Portal.

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# load using pyarrow for performance (crime data is joined between crime & neighborhood & police using geom)
df_crime = pd.read_csv("../Data/chicago_crimes_export.csv", engine="pyarrow", dtype_backend="pyarrow")
# Police Info
police = pd.read_csv("../Data/police_beats_export.csv", engine="pyarrow", dtype_backend="pyarrow")

In [4]:
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate
0,JJ214830,2025-04-03 13:00:00,100XX W BALMORAL AVE,0810,THEFT,OVER $500,I,THEFT,OVER $500,AIRPORT BUILDING NON-TERMINAL - NON-SECURE AREA,f,f,1654,16,41,76,2025,2026-03-14 15:41:39,06,60666,282085138.571,O'Hare,OHARE,371835607.687,16,5,1654,76,OHARE,371835607.687,41.976182,-87.876421,1108491,1934242
1,JJ240422,2025-04-03 13:00:00,021XX W 71ST ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,N,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE - GARAGE,f,t,735,7,17,67,2025,2026-03-14 15:41:39,08B,60636,104114706.716,Englewood,ENGLEWOOD,173600015.009,7,3,735,67,WEST ENGLEWOOD,87947691.9478,41.764765,-87.677703,1163114,1857553
2,JJ205003,2025-04-03 13:00:00,007XX N CENTRAL PARK AVE,0560,ASSAULT,SIMPLE,N,ASSAULT,SIMPLE,RESIDENCE,f,t,1112,11,27,23,2025,2026-03-14 15:41:39,08A,60624,99418122.6738,Humboldt Park,HUMBOLDT PARK,125010425.593,11,2,1121,23,HUMBOLDT PARK,100480876.502,41.894273,-87.716279,1152251,1904667
3,JJ205029,2025-04-03 13:00:00,006XX N WELLS ST,0870,THEFT,POCKET-PICKING,I,THEFT,POCKET-PICKING,STREET,f,f,1832,18,42,8,2025,2026-03-14 15:41:39,06,60654,15869961.5669,River North,RIVER NORTH,38766442.5194,18,3,1832,8,NEAR NORTH SIDE,76675895.9728,41.893645,-87.634123,1174621,1904610
4,JJ204673,2025-04-03 13:00:00,119XX S HALSTED ST,0460,BATTERY,SIMPLE,N,BATTERY,SIMPLE,CTA BUS,f,f,524,5,21,53,2025,2026-03-14 15:41:39,08B,60643,207706232.893,West Pullman,WEST PULLMAN,99365198.0822,5,2,524,53,WEST PULLMAN,99365198.0822,41.67736,-87.641894,1173139,1825780


## Describe Data

In [5]:
df_crime.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
case_number,8543704,8543089,HZ140230,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8543704,NaN,NaN,NaN,2011-09-18 00:52:29,2001-01-01 00:00:00,2005-06-27 19:24:45,2010-07-10 02:00:00,2017-07-28 07:00:00,2026-04-25 00:00:00,NaN
block,8543704,65797,001XX N STATE ST,17201,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,8543704,418,0820,683850,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_description,8524333,32,THEFT,1804998,NaN,NaN,NaN,NaN,NaN,NaN,NaN
secondary_description,8524333,369,SIMPLE,1005413,NaN,NaN,NaN,NaN,NaN,NaN,NaN
index_code,8524333,2,N,5046029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,8543704,34,THEFT,1814892,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,8543704,569,SIMPLE,1005413,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,8527736,218,STREET,2232912,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Wrangle
- Chicago's `IUCR` codes (Illinois Uniform Crime Reporting) are four-digit codes for classifying crimes, with the Chicago Police Department (CPD) using over 400, including FBI Index Offenses (homicide, robbery, theft) and Non-Index offenses (vandalism, weapons violations)
- Chicago has `50 wards`, each represented by an alderperson, with boundaries redrawn every eight years
- The Chicago Police Department (CPD) divides the city into `22 Districts`, which are further broken down into smaller patrol zones called `Beats`, with specific 4-digit numbers for each area
- Chicago is divided into `77 official Community Areas.

In [6]:
df_crime.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8543704 entries, 0 to 8543703
Data columns (total 34 columns):
 #   Column                  Non-Null Count    Dtype                
---  ------                  --------------    -----                
 0   case_number             8543704 non-null  string[pyarrow]      
 1   date                    8543704 non-null  timestamp[s][pyarrow]
 2   block                   8543704 non-null  string[pyarrow]      
 3   iucr                    8543704 non-null  string[pyarrow]      
 4   primary_description     8524333 non-null  string[pyarrow]      
 5   secondary_description   8524333 non-null  string[pyarrow]      
 6   index_code              8524333 non-null  string[pyarrow]      
 7   primary_type            8543704 non-null  string[pyarrow]      
 8   description             8543704 non-null  string[pyarrow]      
 9   location_description    8527736 non-null  string[pyarrow]      
 10  arrest                  8543704 non-null  string[pyarr

In [7]:
# mask operation
mask = df_crime['community_code'] != df_crime['ca_community_code']
# count the difference
mask.sum()
# display
df_crime.loc[mask, ['community_code','ca_community_code']].head()

,community_code,ca_community_code
20,34,60
65,20,23
147,13,14
161,6,3
173,73,72


In [8]:
# mask operation
mask = (
    df_crime['primary_neighborhood'].fillna('').str.upper()
    != df_crime['secondary_neighborhood'].fillna('')
)
# count the difference
print(mask.sum())
# display
df_crime.loc[mask, ['primary_neighborhood','secondary_neighborhood']].sample(n=10)

3162049


,primary_neighborhood,secondary_neighborhood
207088,Mckinley Park,"BRIGHTON PARK,MCKINLEY PARK"
1059080,Irving Park,"IRVING PARK,AVONDALE"
6038847,Brighton Park,"BRIGHTON PARK,MCKINLEY PARK"
1524292,Chicago Lawn,"MARQUETTE PARK,GAGE PARK"
6576831,Brighton Park,"BRIGHTON PARK,MCKINLEY PARK"
4066497,West Town,"WICKER PARK,WEST TOWN"
8338365,East Side,SOUTHEAST SIDE
5017069,Albany Park,"NORTH PARK,ALBANY PARK"
6153788,Archer Heights,"ARCHER HEIGHTS,WEST ELSDON"
193475,South Shore,"SOUTH SHORE, GRAND CROSSING"


### Remove Duplicates

In [9]:
# Remove Duplicate Case Numbers
cleaned_dict = utils.deduplicate_and_report(df_crime)
# Sanity Check
pd.DataFrame(cleaned_dict['df_cleaned']['case_number'].describe()).T

DEDUPLICATION SUMMARY (Arrow Backend)
Original Rows : 8,543,704
Cleaned Rows  : 8,543,089
Total Deleted : 615

TOP 10 DELETIONS BY CASE NUMBER:
case_number  records_deleted
   HJ590004                5
   HZ140230                5
   JC470284                4
   JE266473                4
   HS256531                4
   HP296582                4
   JJ309322                3
   HJ756295                3
   HJ104730                3
   HY346207                3


,count,unique,top,freq
case_number,8543089,8543089,01G050460,1


In [10]:
# Picks 5 values from the cleaned_dict report DataFrame
random_5 = np.random.choice(cleaned_dict['report']['case_number'], size=5, replace=False)
# display
df_crime.loc[df_crime.case_number.isin(random_5),]

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate
603603,JF496175,2022-12-03 01:41:00,009XX W 87TH ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,GARAGE,t,f,2222,22,21,71,2022,2023-05-09 15:45:48,01A,60620,190139582.842,Auburn Gresham,AUBURN GRESHAM,105065353.602,6,1,613,71,AUBURN GRESHAM,105065353.602,41.736038,-87.646167,1171801,1847152
603614,JF496175,2022-12-03 01:10:00,009XX W 87TH ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,GARAGE,t,f,2222,22,21,71,2022,2023-05-09 15:45:48,01A,60620,190139582.842,Auburn Gresham,AUBURN GRESHAM,105065353.602,6,1,613,71,AUBURN GRESHAM,105065353.602,41.736038,-87.646167,1171801,1847152
919200,JE175526,2021-07-26 12:57:00,038XX S MICHIGAN AVE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,STREET,f,f,213,2,3,35,2021,2022-09-19 15:41:05,01A,60653,67759828.1639,Douglas,BRONZEVILLE,46004621.137,2,1,213,35,DOUGLAS,46004621.1581,41.824933,-87.623105,1177828,1879596
987557,JE175526,2021-03-21 18:40:00,038XX S MICHIGAN AVE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,STREET,f,f,213,2,3,35,2021,2022-09-19 15:41:05,01A,60653,67759828.1639,Douglas,BRONZEVILLE,46004621.137,2,1,213,35,DOUGLAS,46004621.1581,41.824933,-87.623105,1177828,1879596
4704605,HP610973,2008-10-06 00:01:00,103XX S AVENUE M,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,HOUSE,t,f,432,4,10,52,2008,2022-09-19 15:41:05,01A,60617,452837420.813,East Side,SOUTHEAST SIDE,83241728.0493,4,3,432,52,EAST SIDE,83241728.0493,41.707648,-87.537704,1201500,1837062
4704612,HP610973,2008-10-06 00:01:00,103XX S AVENUE M,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,HOUSE,t,f,432,4,10,52,2008,2022-09-01 15:42:17,01A,60617,452837420.813,East Side,SOUTHEAST SIDE,83241728.0493,4,3,432,52,EAST SIDE,83241728.0493,41.707648,-87.537704,1201500,1837062
6383470,HK827036,2004-12-25 03:45:00,001XX E 113TH ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,APARTMENT,t,f,531,5,9,49,2004,2022-09-19 15:41:05,01A,60628,345241691.573,Roseland,"WASHINGTON HEIGHTS,ROSELAND",134313706.73,5,3,531,49,ROSELAND,134313706.73,41.689005,-87.619143,1179319,1830074
6383828,HK827036,2004-12-25 00:05:00,001XX E 113TH ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,APARTMENT,t,f,531,5,9,49,2004,2022-09-19 15:41:05,01A,60628,345241691.573,Roseland,"WASHINGTON HEIGHTS,ROSELAND",134313706.73,5,3,531,49,ROSELAND,134313706.73,41.689005,-87.619143,1179319,1830074
6383830,HK827036,2004-12-25 00:05:00,001XX E 113TH ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,APARTMENT,t,f,531,5,9,49,2004,2022-09-19 15:41:05,01A,60628,345241691.573,Roseland,"WASHINGTON HEIGHTS,ROSELAND",134313706.73,5,3,531,49,ROSELAND,134313706.73,41.689005,-87.619143,1179319,1830074
7962890,G540323,2001-09-08 03:56:00,005XX E 89 PLACE,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,AUTO,t,f,633,6,<NA>,<NA>,2001,2022-09-01 15:42:17,01A,60619,167872012.337,Chatham,"CHATHAM,BURNSIDE",82320670.3112,6,3,633,44,CHATHAM,82320670.3112,41.73199,-87.610929,1181431,1845756


### Deep Cleaned Copy

In [11]:
# Deep Copy 
df_crime = cleaned_dict['df_cleaned'].copy()

### The Timeline Definition
* To ensure the analysis is accurate, we define the three eras based on global lockdown patterns:
    * Pre-COVID: January 2001 – February 2020
    * COVID Era: March 2020 – December 31, 2022
    * Post-COVID: January 2023 – Present

In [12]:
# Define boundary timestamps
covid_start      = pd.Timestamp('2020-03-01')
post_covid_start = pd.Timestamp('2023-01-01')

# Define conditions (evaluated top to bottom)
conditions = [
    df_crime['date'] < covid_start,          # Pre-COVID
    df_crime['date'] < post_covid_start      # COVID
]

# Corresponding labels
choices = ['pre_covid', 'covid']

# Apply vectorized selection
df_crime['era'] = (
    np.select(conditions, choices, default='post_covid')
    )

# Cast to memory-efficient categorical dtype
# Only 3 unique values -> int8 index is sufficient
df_crime['era'] = df_crime['era'].astype(arrow_cat8)

# Sanity Check
print(df_crime['era'].value_counts())
print(f"\nEra null count: {df_crime['era'].isna().sum()}")
print(f"\nDate range per era:")
print(df_crime.groupby('era')['date'].agg(['min', 'max']))

era
pre_covid     7092863
post_covid     826157
covid          624069
Name: count, dtype: int64[pyarrow]

Era null count: 0

Date range per era:
                            min                  max
era                                                 
covid       2020-03-01 00:00:00  2022-12-31 23:55:00
post_covid  2023-01-01 00:00:00  2026-04-25 00:00:00
pre_covid   2001-01-01 00:00:00  2020-02-29 23:59:00


### Invalid (x_coordinate, y_coordinate, latitude, longitude)

In [13]:
# Condition 1: Invalid State Plane coordinates
mask_xy = (df_crime.x_coordinate == 0) | (df_crime.y_coordinate == 0)

# Standard Chicago Bounding Box (WGS84)
LAT_MIN, LAT_MAX = 41.60, 42.05
LON_MIN, LON_MAX = -87.94, -87.50

# Condition 2: Invalid WGS84 coordinates (outside Chicago bounding box)
mask_latlon = (
    (df_crime.latitude  < LAT_MIN) |
    (df_crime.latitude  > LAT_MAX) |
    (df_crime.longitude < LON_MIN) |
    (df_crime.longitude > LON_MAX)
)

# Diagnostic
print(f"Invalid x/y:      {mask_xy.sum():,}")
print(f"Invalid lat/lon:  {mask_latlon.sum():,}")
print(f"Overlap:          {(mask_xy & mask_latlon).sum():,}")
print(f"Total unique:     {(mask_xy | mask_latlon).sum():,}")

# Null out invalid coordinates (float cols → np.nan)
df_crime.loc[mask_xy,     ['x_coordinate', 'y_coordinate']] = np.nan
df_crime.loc[mask_latlon, ['latitude', 'longitude']]        = np.nan

Invalid x/y:      149
Invalid lat/lon:  149
Overlap:          149
Total unique:     149


### Invalid community_code

In [14]:
# Invalid community_code
mask = (df_crime.community_code == 0)
mask.sum()

76

In [15]:
# Update community_code
df_crime.loc[mask, ['community_code']] = pd.NA

### Check for Duplicates

In [16]:
# get dupes
dupes = df_crime.duplicated(keep='last')
# any duplicates
if dupes.any():
    print(f"Number of Duplicates: {dupes.sum():,}")
else:
    print("No Duplicates")

No Duplicates


### Unique Values

In [17]:
# display number of unique values
for i in df_crime.columns:
    print(f"{i}: {df_crime[i].nunique():,}")

case_number: 8,543,089
date: 3,584,258
block: 65,796
iucr: 418
primary_description: 32
secondary_description: 369
index_code: 2
primary_type: 34
description: 569
location_description: 218
arrest: 2
domestic: 2
beat: 305
district: 24
ward: 50
community_code: 77
year: 26
updated_on: 7,658
fbi_code: 26
zip_code: 59
zip_code_area: 59
primary_neighborhood: 98
secondary_neighborhood: 78
neighborhood_area: 98
p_district: 23
p_sector: 5
p_beat: 275
ca_community_code: 77
ca_community_name: 77
ca_community_area: 77
latitude: 911,676
longitude: 911,080
x_coordinate: 79,380
y_coordinate: 130,455
era: 3


### Update: location_description

In [18]:
# Replaces : , -, and multi-spaces with a single space
def clean_locations(series):
    out = (
        series.str.replace(r'[\s:,-]+', ' ', regex=True)  # Combined delimiters to space
              .str.replace(r'\s*/\s*', '/', regex=True)   # Fix slashes
              .str.strip()
    )
    
    return out
# Apply regex
df_crime['location_description'] = clean_locations(df_crime['location_description'])

# Dictionary mapping (Vectorized replace)
mapping = {
    'NURSING HOME/RETIREMENT HOME': 'NURSING/RETIREMENT HOME', 
    'OTHER RAILROAD PROP/TRAIN DEPOT': 'OTHER RAILROAD PROPERTY/TRAIN DEPOT',
    'PARKING LOT/GARAGE(NON.RESID.)': 'PARKING LOT/GARAGE (NON RESIDENTIAL)',
    'POLICE FACILITY/VEH PARKING LOT': 'POLICE FACILITY/VEHICLE PARKING LOT',
    'POOLROOM': 'POOL ROOM', 
    'RESIDENCE YARD (FRONT/BACK)': 'RESIDENTIAL YARD (FRONT/BACK)',
    'TAXICAB': 'TAXI CAB',
    'VEHICLE OTHER RIDE SERVICE': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)',
    'VEHICLE OTHER RIDE SHARE SERVICE (E.G. UBER LYFT)': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)'
}

# Apply mapping first
df_crime['location_description'] = df_crime['location_description'].replace(mapping)

### New Feature(s)

In [19]:
# Add Month & Day of the Week
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Extract integers and map them
# .dt.month returns 1-12, so we subtract 1 for 0-based indexing
df_crime['month'] = np.array(months)[df_crime['date'].dt.month.values - 1]

# .dt.dayofweek returns 0-6 (0 is Monday)
df_crime['day_of_week'] = np.array(days)[df_crime['date'].dt.dayofweek.values]

# convert to pyarrow
df_crime['month'] = df_crime['month'].astype(arrow_string)
df_crime['day_of_week'] = df_crime['day_of_week'].astype(arrow_string)

In [20]:
# Map to strings first
df_crime['quarter'] = df_crime['date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}).astype(arrow_string)
# combine
df_crime['year_quarter'] = (df_crime['year'].astype("string[pyarrow]") + "-" + df_crime['quarter'])
# Force pyarrow datatype
df_crime['year_quarter'] = df_crime['year_quarter'].astype(arrow_string)

| Interval (Inclusive, Exclusive) | Mathematical Notation | Label        | Hours Included    |
|--------------------------------|----------------------|--------------|-------------------|
| 1st: 0 to 4                    | \([0, 4)\)          | Late Night   | 0, 1, 2, 3       |
| 2nd: 4 to 8                    | \([4, 8)\)          | Early Morning| 4, 5, 6, 7       |
| 3rd: 8 to 12                   | \([8, 12)\)         | Morning      | 8, 9, 10, 11     |
| 4th: 12 to 16                  | \([12, 16)\)        | Afternoon    | 12, 13, 14, 15   |
| 5th: 16 to 20                  | \([16, 20)\)        | Evening      | 16, 17, 18, 19   |
| 6th: 20 to 24                  | \([20, 24)\)        | Night        | 20, 21, 22, 23   |

In [21]:
# Get the hours as a PyArrow-backed integer
hours = df_crime['date'].dt.hour.values

# Use np.digitize for ultra-fast binning (vectorized)
# bins: [0, 4, 8, 12, 16, 20, 24]
# digitize returns 1 for 0-3, 2 for 4-7, etc.
bin_indices = np.digitize(hours, bins=[4, 8, 12, 16, 20])

# Map indices to labels
time_labels = np.array(['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night'])
df_crime['time_of_day'] = time_labels[bin_indices]

# Final cast to string[pyarrow]
df_crime['time_of_day'] = df_crime['time_of_day'].astype(arrow_string)

### FBI Code Mapping

In [22]:
# display fbi_code
print(sorted(df_crime['fbi_code'].unique()))

['01A', '01B', '02', '03', '04A', '04B', '05', '06', '07', '08A', '08B', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '22', '24', '26']


In [23]:
# determine specific values in the data that are missing from the mapping dictionary
df_crime.loc[~df_crime["fbi_code"].isin(geo_dict.fbi_codes.keys()), "fbi_code" ].unique()

<ArrowExtensionArray>
[]
Length: 0, dtype: string[pyarrow]

In [24]:
df_crime[["fbi_code_desc", "fbi_index_code"]] = pd.DataFrame({
    "fbi_code_desc": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["desc"]),
    "fbi_index_code": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["is_index"])
})

# convert to arrow datatype
df_crime["fbi_code_desc"] = df_crime["fbi_code_desc"].astype(arrow_string)
df_crime["fbi_index_code"] = df_crime["fbi_index_code"]

In [25]:
# df_crime[['iucr','primary_description','secondary_description','description','fbi_code']][df_crime['fbi_code'] == '01A'].sample(5)
df_crime[['fbi_code_desc','description', 'location_description', 'domestic', 'fbi_index_code']].sample(n=5, random_state=_RANDOM_STATE)

,fbi_code_desc,description,location_description,domestic,fbi_index_code
6005870,Miscellaneous Non-Index Offenses,ANIMAL ABUSE/NEGLECT,OTHER,f,False
2904737,Vandalism,TO PROPERTY,RESIDENTIAL YARD (FRONT/BACK),f,False
5835722,Simple Battery,SIMPLE,STREET,f,False
3503885,Larceny – Theft,$500 AND UNDER,CHA APARTMENT,f,True
4490856,Drug Abuse Violations,POSS: CANNABIS 30GMS OR LESS,ALLEY,f,False


### Datatype Change (Boolean)

In [26]:
# check for unique values
df_crime[['arrest','domestic', 'fbi_index_code']].apply(lambda s: s.unique())

,arrest,domestic,fbi_index_code
0,t,f,False
1,f,t,True


In [27]:
# convert to pyarrow boolean
df_crime[['arrest','domestic']] = (
    df_crime[['arrest','domestic']] # domestic: Domestic violence
        .apply(lambda col: col.map({'t': True, 'f': False}))
        .astype(bool)
)

### Feature Information:
* A Chicago `ward` is one of 50 legislative districts, each represented by an elected Alderman on the City Council, serving as local government branches to provide city services, manage development, and reflect community demographics, with boundaries redrawn every 10 years based on census data.
* The `district` feature refers to the city's 22 police districts, which are geographic areas used to organize crime data.
* The `beat` feature in Chicago crime data identifies the smallest geographic police area (a beat) where a crime occurred.
* The `sector` refers to a specific geographic division used by the Chicago Police Department (CPD), where several smaller `beats` (police patrol areas) are grouped together to form a sector, which then rolls up into a larger `district`, providing a layered geographic context for analyzing crime trends.
* The `Community Area` feature refers to one of 77 distinct, officially defined, and geographically stable neighborhoods used for urban planning and statistical analysis. This feature allows categorizing crime incidents by location, enabling trend analysis and identifying high-crime areas.

In [28]:
# Remove columns (Assign the result back to the main variable)
remove_cols = ["p_district", "p_beat"]
df_crime = df_crime.drop(columns=remove_cols)

# Rename 'p_sector' to 'sector'
df_crime = df_crime.rename(columns={'p_sector': 'sector'})

# Verify the change
print(df_crime.columns)

Index(['case_number', 'date', 'block', 'iucr', 'primary_description',
       'secondary_description', 'index_code', 'primary_type', 'description',
       'location_description', 'arrest', 'domestic', 'beat', 'district',
       'ward', 'community_code', 'year', 'updated_on', 'fbi_code', 'zip_code',
       'zip_code_area', 'primary_neighborhood', 'secondary_neighborhood',
       'neighborhood_area', 'sector', 'ca_community_code', 'ca_community_name',
       'ca_community_area', 'latitude', 'longitude', 'x_coordinate',
       'y_coordinate', 'era', 'month', 'day_of_week', 'quarter',
       'year_quarter', 'time_of_day', 'fbi_code_desc', 'fbi_index_code'],
      dtype='object')


### Update Datatypes & Fill
- Add padding if required

In [29]:
# display
police.head()

,district,sector,beat
0,1,1,111
1,1,1,112
2,1,1,113
3,1,1,114
4,1,2,121


In [30]:
# change to string and must be three char length
cols = police.columns.to_list()

# iterate cols
for col in cols:

    if col in ['district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        police[col] = police[col].astype("string").str.zfill(3)
    elif col in ['beat']:
         police[col] = police[col].astype("string").str.zfill(4)
    else:
        # Fill NAs and convert to a standard string
        police[col] = police[col].astype("string")
        
    # Force ArrowDtype
    police[col] = police[col].astype(arrow_string)

police[cols].sample(5)

,district,sector,beat
249,022,3,2234
132,011,1,1115
159,014,1,1412
190,016,3,1634
251,024,1,2412


In [31]:
# change to string and must be three char length
cols = ['district', 'beat', 'ward', 'sector', 'community_code', 'year']

# iterate cols
for col in cols:

    if col in ['district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        df_crime[col] = df_crime[col].astype("string").str.zfill(3)
    elif col in ['ward', 'community_code']:
        df_crime[col] = df_crime[col].astype("string").str.zfill(2)
    elif col in ['beat']:
         df_crime[col] = df_crime[col].astype("string").str.zfill(4)
    else:
        # Fill NAs and convert to a standard string
        df_crime[col] = df_crime[col].astype("string")
        
    # Force ArrowDtype
    df_crime[col] = df_crime[col].astype(arrow_string)

df_crime[cols].sample(5)

,district,beat,ward,sector,community_code,year
2547149,007,0714,15,1,67,2015
987699,022,2233,34,3,49,2021
2766227,012,1212,01,1,24,2014
2376005,010,1031,22,3,30,2015
2400099,003,0314,20,1,42,2015


### Update Invalid district / New column (district_loc)

In [32]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 021, 022, 024, 025, 031]
::::: Unique Count: 24 (+ 47 nulls)


In [33]:
# Invalid district (21, 31)
mask = df_crime.district.isin(['021', '031'])
mask.sum()

np.int64(281)

In [34]:
# Update Invalid district
df_crime.loc[mask, 'district'] = pd.NA 

In [35]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22 (+ 328 nulls)


In [36]:
utils.wrap_unique(police, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22


In [37]:
utils.wrap_unique(police, 'sector')

[1, 2, 3, 5]
::::: Unique Count: 4


In [38]:
# display sector
utils.wrap_unique(df_crime, 'sector')

[0, 1, 2, 3, 5]
::::: Unique Count: 5 (+ 117,393 nulls)


According to a search, the Chicago Police Department (CPD) currently operates 277 active, specialized beats across 22 districts, employing a community-policing model in which 8–9 officers are assigned to patrol specific areas for at least a year. Our Police table in the Chicago Data Hub lists 274 beats, but our crime data contains 305, and we assume that some beats were consolidated due to Chuicago Police Department overhaul over the past 20 years.

In [39]:
print("Number of Beats in the Crime Data:" , df_crime.beat.nunique())
print("Number of Beats in the Police Data:" , police.beat.nunique())

Number of Beats in the Crime Data: 305
Number of Beats in the Police Data: 274


In [40]:
# Compare
crime_set = set(df_crime.beat.to_list())
police_set = set(police.beat.to_list())

# check sets
print("Does crime_set contain all of police_set: " ,crime_set.issuperset(police_set))
print("Does police_set contain all of crime_set: " ,police_set.issuperset(crime_set))

Does crime_set contain all of police_set:  True
Does police_set contain all of crime_set:  False


In [41]:
# Set Compare (symmetric difference)
print(sorted(crime_set ^ police_set))
print("Mis-match Beat Count:", len(crime_set ^ police_set))

['0134', '0310', '0430', '1311', '1312', '1313', '1322', '1323', '1324', '1331', '1332', '1333', '1650', '2111', '2112', '2113', '2122', '2123', '2124', '2131', '2132', '2133', '2311', '2312', '2313', '2322', '2323', '2324', '2331', '2332', '2333']
Mis-match Beat Count: 31


In [42]:
print(sorted(crime_set - police_set))
print("Extra Beat Count from crime_set:", len(crime_set ^ police_set))

['0134', '0310', '0430', '1311', '1312', '1313', '1322', '1323', '1324', '1331', '1332', '1333', '1650', '2111', '2112', '2113', '2122', '2123', '2124', '2131', '2132', '2133', '2311', '2312', '2313', '2322', '2323', '2324', '2331', '2332', '2333']
Extra Beat Count from crime_set: 31


In [43]:
# Initialize
invalid_beat = list(crime_set - police_set)

In [44]:
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,sector,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
8265318,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSSESS - CANNABIS 30 GRAMS OR LESS,N,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,<NA>,<NA>,2001,2015-08-17 15:03:40,18,60649,80526075.8505,South Shore,"SOUTH SHORE, GRAND CROSSING",81812716.3904,2,43,SOUTH SHORE,81812716.3958,41.764219,-87.582549,1189075,1857566,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False
7080787,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,I,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,08,69,2003,2015-08-17 15:03:40,06,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True
6396203,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSSESS - HEROIN (WHITE),N,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,27,23,2004,2018-02-28 15:56:25,18,60624,99418122.6738,Humboldt Park,HUMBOLDT PARK,125010425.593,2,23,HUMBOLDT PARK,100480876.502,41.892451,-87.719888,1151273,1903996,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False
5818733,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,I,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,29,19,2006,2018-02-28 15:56:25,05,60707,48519709.6539,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,1,19,BELMONT CRAGIN,109099414.689,41.928096,-87.78561,1133296,1916864,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True
5312174,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSSESS - CANNABIS MORE THAN 30 GRAMS,N,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,31,19,2007,2018-02-28 15:56:25,18,60639,127476051.26,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,2,19,BELMONT CRAGIN,109099414.689,41.921066,-87.747452,1143697,1914371,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False


In [45]:
# columns to display
cols = ['year', 'primary_neighborhood',	'secondary_neighborhood', 'neighborhood_area', 'community_code', 
        'ca_community_code','ca_community_name', 'ca_community_area', 'ward', 'district', 'sector', 'beat',
        'zip_code', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']
mask = df_crime.beat.isin(invalid_beat)
print(f"Possible Invalid 'beat' from Cime Data Count:  {(mask.sum()):,}")
bad_beat = df_crime.loc[mask, cols].copy()
bad_beat.head()

Possible Invalid 'beat' from Cime Data Count:  351,327


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8272314,2001,Douglas,BRONZEVILLE,46004621.137,<NA>,35,DOUGLAS,46004621.1581,<NA>,002,1,2112,60616,1178109,1882924,41.834059,-87.621973
7918746,2001,Oakland,"OAKLAND,KENWOOD",16913961.0408,<NA>,36,OAKLAND,16913961.0408,<NA>,002,1,2123,60653,1183840,1878113,41.820725,-87.601095
8010810,2001,Kenwood,"KENWOOD,OAKLAND",29071741.9283,39,39,KENWOOD,29071741.9283,04,002,2,2124,60615,1182421,1872504,41.805367,-87.606475
7906901,2001,Near South Side,NEAR SOUTH SIDE,34252582.7003,<NA>,33,NEAR SOUTH SIDE,49769639.4541,<NA>,002,3,2111,60616,1177107,1890696,41.855409,-87.625414
8294051,2001,Uptown,UPTOWN,65095642.836,<NA>,3,UPTOWN,65095642.7289,<NA>,019,1,2311,60640,1167921,1930894,41.965917,-87.657969


In [46]:
# count rows by year
bad_beat['year'].value_counts().sort_index()

year
2001    36982
2002    36251
2003    35330
2004    34972
2005    34714
2006    32333
2007    30002
2008    29480
2009    25284
2010    24096
2011    22549
2012     9201
2020        1
2023       91
2024       41
Name: count, dtype: int64[pyarrow]

In [47]:
mask = bad_beat.year.isin(['2024', '2023', '2020'])
bad_beat.loc[mask, cols].drop_duplicates().head()

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
528421,2023,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1650,60666,1108491,1934242,41.976182,-87.876421
1239683,2020,<NA>,<NA>,<NA>,76,<NA>,<NA>,<NA>,41,016,<NA>,1650,<NA>,<NA>,<NA>,<NA>,<NA>
250619,2023,<NA>,<NA>,<NA>,76,<NA>,<NA>,<NA>,41,016,<NA>,1650,<NA>,<NA>,<NA>,<NA>,<NA>
307120,2024,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1650,60666,1108491,1934242,41.976182,-87.876421
298874,2024,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1650,60666,1108424,1934249,41.976202,-87.876667


##### According to [Chicago District Map](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://chicagocop.com/wp-content/uploads/Chicago-Police-Department-Citywide-Area-District-Beat-Map-2009-March.pdf), beat 1650 does not exist; it's possible it was merged into beat 1651, since that is the O'Hare community. We will not change the beat column and will assume it is accurate, since we are unable to verify from the source whether any changes or consolidations occurred with the Chicago Police Department overall.

### **Note:**
* Chicago’s crime data is recorded across a complex framework of overlapping jurisdictions, ranging from political districts to social neighborhoods. At the administrative level, the Chicago Police Department operates through a hierarchy of Districts and Beats. A Beat is the smallest geographic unit, assigned to a specific patrol car for community policing, while multiple Beats are grouped into a District managed by a central precinct. For example, Beats 2511, 2514, and 2521 all fall under the jurisdiction of District 025. Because these boundaries are drawn based on population density and response times rather than cultural history, they rarely align perfectly with the city’s social fabric.
* To provide a more stable lens for analysis, researchers utilize the city’s 77 Community Areas. Established in the 1920s by the University of Chicago, these fixed boundaries remain unchanged by political redistricting or postal updates, allowing for consistent longitudinal tracking of crime trends over decades. In contrast, Chicago’s 50 Wards are political entities redrawn every ten years to ensure equal population representation.
* Ultimately, "neighborhood" designations like "Albany Park" or "Irving Park" reflect social and historical identities rather than law-enforcement jurisdictions. Because these residential areas are often too large for a single patrol car to cover, a single neighborhood is frequently split across multiple Police Beats. This misalignment means that a single criminal incident may be categorized differently depending on whether the analyst is looking through a political (Ward), statistical (Community Area), or operational (Police District) lens.

### **NaN Note:**
- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation by using data from other sources to fill the corresponding NaN entries in the location-based fields.

#### Update sector

In [48]:
# Invert the mapping
cpd_sector = {
    dist: sector
    for sector, dists in geo_dict.cpd_sector.items() # Outer loop: looping over sector: list_of_districts
    for dist in dists # Inner loop: looping over each district inside that list
}

# Intentional overwrite: sector is derived deterministically from district
# using the cpd_sector dictionary (source:https://www.chicagopolice.org/statistics-data/crime-statistics/)
# sector is represented as Area
# update sector (note: it's not correct for all the years)
df_crime['sector'] = (
    df_crime['district']
        .map(cpd_sector)
).astype(arrow_cat8)

In [49]:
# New column
df_crime['district_location'] = df_crime.district.map(geo_dict.cpd_districts)
df_crime['district_location'] = df_crime['district_location'].astype(arrow_cat8)

### NaNs

In [50]:
# display
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                         Count Percentage
ward                    614800    7.1965%
community_code          613787    7.1846%
secondary_neighborhood  118627    1.3886%
ca_community_name       118627    1.3886%
primary_neighborhood    118627    1.3886%
neighborhood_area       118627    1.3886%
ca_community_area       118627    1.3886%
ca_community_code       118627    1.3886%
zip_code                118568    1.3879%
zip_code_area           118568    1.3879%
y_coordinate             96633    1.1311%
x_coordinate             96633    1.1311%
longitude                96633    1.1311%
latitude                 96633    1.1311%
primary_description      19371    0.2267%
secondary_description    19371    0.2267%
index_code               19371    0.2267%
location_description     15968    0.1869%
sector                     328    0.0038%
district                   328    0.0038%
district_location          328    0.0038%


### Inital Impute

In [51]:
def get_sample_report(data_df: pd.DataFrame, bool_mask: pd.Series, cols: list=cols, n_sample: int = 5, seed: int = _RANDOM_STATE):
    """
    Displays a sample of the dataframe based on a mask.
    If there are fewer than 5 available rows, it displays all. Otherwise, displays n_sample.
    """
    # 1. Filter the data based on the mask
    filtered_df = data_df.loc[bool_mask, cols]
    available_count = len(filtered_df)

    # 2. Display counts from the mask
    print(f"::::: Missing: {(bool_mask.sum()):,} :::::\n")
    
    
    # 3. Logic: If count < 5, take all. Else, take n_sample (default 5)
    if available_count < 5:
        sample_df = filtered_df
    else:
        sample_df = filtered_df.sample(n=min(n_sample, available_count), random_state=seed)
    
    # 4. Capture index and display
    sample_indices = sample_df.index.tolist()
    
    print(f"--- Showing {len(sample_df)} of {available_count} eligible rows ---")

    return sample_indices

#### ward

In [52]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
        df_crime.ward.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 605,511 :::::

--- Showing 5 of 605511 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8259144,2001,Grand Crossing,"SOUTH SHORE, GRAND CROSSING",98853167.7093,<NA>,69,GREATER GRAND CROSSING,98853167.7093,<NA>,007,1,0731,60621,1175937,1856141,41.760613,-87.630745
8067369,2001,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,<NA>,19,BELMONT CRAGIN,109099414.689,<NA>,025,5,2521,60639,1141537,1917427,41.929492,-87.755313
7868494,2001,Beverly,BEVERLY,88779363.9384,<NA>,72,BEVERLY,88779363.9384,<NA>,022,2,2212,60643,1162267,1835572,41.704464,-87.681418
8252539,2001,Rogers Park,ROGERS PARK,51259902.4506,<NA>,1,ROGERS PARK,51259902.4506,<NA>,024,3,2422,60626,1163959,1950960,42.021064,-87.671967
7687361,2002,Englewood,ENGLEWOOD,173600015.009,<NA>,68,ENGLEWOOD,85652323.0826,<NA>,007,1,0732,60621,1173861,1857638,41.764767,-87.63831


In [53]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'community_code', 'district', 'sector', 'beat', 'zip_code']
update_cols = ['ward']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 605,511
Rows updated:       605,455
Rows not updated:   56


In [54]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8259144,2001,Grand Crossing,"SOUTH SHORE, GRAND CROSSING",98853167.7093,<NA>,69,GREATER GRAND CROSSING,98853167.7093,08,007,1,0731,60621,1175937,1856141,41.760613,-87.630745
8067369,2001,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,<NA>,19,BELMONT CRAGIN,109099414.689,08,025,5,2521,60639,1141537,1917427,41.929492,-87.755313
7868494,2001,Beverly,BEVERLY,88779363.9384,<NA>,72,BEVERLY,88779363.9384,08,022,2,2212,60643,1162267,1835572,41.704464,-87.681418
8252539,2001,Rogers Park,ROGERS PARK,51259902.4506,<NA>,1,ROGERS PARK,51259902.4506,08,024,3,2422,60626,1163959,1950960,42.021064,-87.671967
7687361,2002,Englewood,ENGLEWOOD,173600015.009,<NA>,68,ENGLEWOOD,85652323.0826,08,007,1,0732,60621,1173861,1857638,41.764767,-87.63831


#### community_code

In [55]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
        df_crime.community_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 604,537 :::::

--- Showing 5 of 604537 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8044534,2001,O'Hare,OHARE,371835607.687,<NA>,76,OHARE,371835607.687,08,016,5,1614,60656,1119276,1935258,41.978805,-87.836738
7757024,2002,Chatham,"CHATHAM,BURNSIDE",82320670.3112,<NA>,44,CHATHAM,82320670.3112,08,006,2,0631,60619,1184621,1851903,41.748784,-87.599051
7915356,2001,Englewood,ENGLEWOOD,173600015.009,<NA>,67,WEST ENGLEWOOD,87947691.9478,08,007,1,0734,60636,1167840,1857620,41.764849,-87.660379
8044528,2001,Jefferson Park,JEFFERSON PARK,64868161.6819,<NA>,11,JEFFERSON PARK,64868161.6818,08,016,5,1623,60646,1138568,1937103,41.983539,-87.765744
8123503,2001,Austin,AUSTIN,170037750.826,<NA>,25,AUSTIN,199254203.427,08,015,4,1513,60644,1137190,1894294,41.866092,-87.771843


In [56]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'district', 'sector', 'beat', 'zip_code']
update_cols = ['community_code']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 604,537
Rows updated:       100,863
Rows not updated:   503,674


#### primary_neighborhood & neighborhood_area & secondary_neighborhood

In [57]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
    (
        df_crime.primary_neighborhood.isna() |
        df_crime.neighborhood_area.isna() |
        df_crime.secondary_neighborhood.isna()
    )
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 21,994 :::::

--- Showing 5 of 21994 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5317778,2007,<NA>,<NA>,<NA>,25,<NA>,<NA>,<NA>,36,025,5,2512,<NA>,1127710,1914568,41.921891,-87.806188
6682349,2004,<NA>,<NA>,<NA>,72,<NA>,<NA>,<NA>,19,022,2,2213,<NA>,1162115,1839281,41.714645,-87.681871
243300,2024,<NA>,<NA>,<NA>,75,<NA>,<NA>,<NA>,21,022,2,2234,<NA>,1166466,1825759,41.677447,-87.66632
1544551,2018,<NA>,<NA>,<NA>,70,<NA>,<NA>,<NA>,18,008,1,0834,<NA>,1156233,1846716,41.735168,-87.703215
6560104,2004,<NA>,<NA>,<NA>,13,<NA>,<NA>,<NA>,50,017,5,1711,<NA>,1151706,1942204,41.997288,-87.71729


In [58]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude',
        'district', 'sector', 'beat']
update_cols = ['secondary_neighborhood', 'primary_neighborhood', 'neighborhood_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 21,994
Rows updated:       129
Rows not updated:   21,865
Values imputed:     387


#### ca_community_code & ca_community_name & ca_community_area

In [59]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
        (
            df_crime.ca_community_code.isna() |
             df_crime.ca_community_name.isna() |
             df_crime.ca_community_area.isna()
        )
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 21,994 :::::

--- Showing 5 of 21994 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5317778,2007,<NA>,<NA>,<NA>,25,<NA>,<NA>,<NA>,36,025,5,2512,<NA>,1127710,1914568,41.921891,-87.806188
6682349,2004,<NA>,<NA>,<NA>,72,<NA>,<NA>,<NA>,19,022,2,2213,<NA>,1162115,1839281,41.714645,-87.681871
243300,2024,<NA>,<NA>,<NA>,75,<NA>,<NA>,<NA>,21,022,2,2234,<NA>,1166466,1825759,41.677447,-87.66632
1544551,2018,<NA>,<NA>,<NA>,70,<NA>,<NA>,<NA>,18,008,1,0834,<NA>,1156233,1846716,41.735168,-87.703215
6560104,2004,<NA>,<NA>,<NA>,13,<NA>,<NA>,<NA>,50,017,5,1711,<NA>,1151706,1942204,41.997288,-87.71729


In [60]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'zip_code', 'district', 'sector', 'beat']
update_cols = ['ca_community_code', 'ca_community_name', 'ca_community_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 21,994
Rows updated:       21,920
Rows not updated:   74
Values imputed:     65,760


In [61]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5317778,2007,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,36,025,5,2512,<NA>,1127710,1914568,41.921891,-87.806188
6682349,2004,<NA>,<NA>,<NA>,72,76,OHARE,371835607.687,19,022,2,2213,<NA>,1162115,1839281,41.714645,-87.681871
243300,2024,<NA>,<NA>,<NA>,75,76,OHARE,371835607.687,21,022,2,2234,<NA>,1166466,1825759,41.677447,-87.66632
1544551,2018,<NA>,<NA>,<NA>,70,76,OHARE,371835607.687,18,008,1,0834,<NA>,1156233,1846716,41.735168,-87.703215
6560104,2004,<NA>,<NA>,<NA>,13,76,OHARE,371835607.687,50,017,5,1711,<NA>,1151706,1942204,41.997288,-87.71729


#### zip_code & zip_code_area

In [62]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.latitude.notna() &
        df_crime.longitude.notna() & 
            df_crime.zip_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 21,935 :::::

--- Showing 5 of 21935 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
3109652,2013,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,29,015,4,1511,<NA>,1136203,1905464,41.896761,-87.7752
682829,2022,<NA>,<NA>,<NA>,01,76,OHARE,371835607.687,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049
3575493,2011,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,36,025,5,2513,<NA>,1127853,1909983,41.909307,-87.805767
6515487,2004,<NA>,<NA>,<NA>,09,76,OHARE,371835607.687,41,016,5,1611,<NA>,1127379,1943526,42.001361,-87.806751
4164681,2010,<NA>,<NA>,<NA>,70,76,OHARE,371835607.687,18,008,1,0834,<NA>,1155555,1846697,41.73513,-87.705699


In [63]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'district', 'sector', 'beat']
update_cols = ['zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 21,935
Rows updated:       129
Rows not updated:   21,806
Values imputed:     258


In [64]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
3109652,2013,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,29,015,4,1511,<NA>,1136203,1905464,41.896761,-87.7752
682829,2022,<NA>,<NA>,<NA>,01,76,OHARE,371835607.687,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049
3575493,2011,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,36,025,5,2513,<NA>,1127853,1909983,41.909307,-87.805767
6515487,2004,<NA>,<NA>,<NA>,09,76,OHARE,371835607.687,41,016,5,1611,<NA>,1127379,1943526,42.001361,-87.806751
4164681,2010,<NA>,<NA>,<NA>,70,76,OHARE,371835607.687,18,008,1,0834,<NA>,1155555,1846697,41.73513,-87.705699


#### sector & district

In [65]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &   
        df_crime.sector.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 328 :::::

--- Showing 5 of 328 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
22610,2025,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,<NA>,<NA>,1653,60666,1107233,1929935,41.964381,-87.881131
3753613,2011,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,08,<NA>,<NA>,1654,60656,1107552,1930183,41.965057,-87.879953
4616577,2008,South Shore,"SOUTH SHORE, GRAND CROSSING",81812716.3904,43,43,SOUTH SHORE,81812716.3958,07,<NA>,<NA>,0421,60649,1196103,1854882,41.756682,-87.556879
4672190,2008,Woodlawn,WOODLAWN,40515739.083,42,42,WOODLAWN,57815179.512,20,<NA>,<NA>,0312,60637,1182746,1863414,41.780415,-87.605565
5983000,2005,Morgan Park,"MOUNT GREENWOOD,MORGAN PARK",91877340.6988,75,75,MORGAN PARK,91877340.6988,19,<NA>,<NA>,2212,60655,1159523,1830893,41.69168,-87.691593


In [66]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'district', 'zip_code', 'beat']
update_cols = ['sector', 'district']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 328
Rows updated:       328
Rows not updated:   0
Values imputed:     656


### Second Pass: Loosen Composite Key

In [67]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                         Count Percentage
community_code          512924    6.0040%
secondary_neighborhood  118498    1.3871%
primary_neighborhood    118498    1.3871%
neighborhood_area       118498    1.3871%
zip_code                118439    1.3864%
zip_code_area           118439    1.3864%
ca_community_area        96707    1.1320%
ca_community_code        96707    1.1320%
ca_community_name        96707    1.1320%
y_coordinate             96633    1.1311%
x_coordinate             96633    1.1311%
longitude                96633    1.1311%
latitude                 96633    1.1311%
primary_description      19371    0.2267%
secondary_description    19371    0.2267%
index_code               19371    0.2267%
location_description     15968    0.1869%
ward                      9345    0.1094%
district_location          328    0.0038%


#### community_code

In [68]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
    df_crime.ward.notna() &
    df_crime.zip_code.notna() &
        df_crime.community_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 503,674 :::::

--- Showing 5 of 503674 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
7901361,2001,Chicago Lawn,"MARQUETTE PARK,GAGE PARK",98279465.1151,<NA>,66,CHICAGO LAWN,98279465.1151,08,008,1,0825,60629,1161369,1864411,41.783621,-87.683909
7876955,2001,Galewood,"MONTCLARE, GALEWOOD",29257505.72,<NA>,25,AUSTIN,199254203.427,08,025,5,2513,60707,1132090,1911386,41.913085,-87.790169
7989734,2001,Humboldt Park,HUMBOLDT PARK,125010425.593,<NA>,23,HUMBOLDT PARK,100480876.502,08,011,4,1123,60624,1153351,1902539,41.888411,-87.712295
7851844,2001,River North,RIVER NORTH,38766442.5194,<NA>,8,NEAR NORTH SIDE,76675895.9728,08,018,3,1824,60610,1174552,1907813,41.902436,-87.63428
8203743,2001,United Center,UNITED CENTER,32520512.7053,<NA>,28,NEAR WEST SIDE,158492466.554,08,012,3,1211,60612,1163721,1900003,41.88124,-87.674285


In [69]:
# Composite key
keys = ['year', 'x_coordinate', 'secondary_neighborhood', 'y_coordinate', 'latitude', 'longitude',
        'primary_neighborhood']
update_cols = ['community_code']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 503,674
Rows updated:       4,857
Rows not updated:   498,817


#### sector & district

In [70]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna() &
    (
        df_crime.sector.isna() |
        df_crime.district.isna()
    )
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 0 :::::

--- Showing 0 of 0 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [71]:
# Composite key
keys = ['x_coordinate', 'secondary_neighborhood', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['sector', 'district']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0
Values imputed:     0


#### ca_community_name

In [72]:
# mask
mask = (
    df_crime.x_coordinate.notna() & 
    df_crime.y_coordinate.notna() &
    df_crime.latitude.notna() &
    df_crime.longitude.notna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 8,446,456 :::::

--- Showing 5 of 8446456 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
356908,2023,Austin,AUSTIN,170037750.826,25,25,AUSTIN,199254203.427,28,011,4,1113,60644,1145364,1900573,41.883171,-87.741677
1614099,2018,Chicago Lawn,"MARQUETTE PARK,GAGE PARK",98279465.1151,66,66,CHICAGO LAWN,98279465.1151,16,008,1,0824,60629,1156734,1865013,41.785368,-87.700886
1769584,2017,Chatham,"CHATHAM,BURNSIDE",82320670.3112,44,44,CHATHAM,82320670.3112,21,006,2,0622,60620,1176740,1847273,41.73626,-87.628069
8132929,2001,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1651,60666,1100635,1934208,41.9762,-87.905312
2374743,2015,Lake View,LAKE VIEW,76561177.2562,06,6,LAKE VIEW,87214799.2728,47,019,3,1912,60613,1164934,1926849,41.954882,-87.669067


In [73]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['ca_community_code', 'ca_community_name', 'ca_community_area']
# Update
df_crime1 = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,446,456
Rows updated:       0
Rows not updated:   8,446,456
Values imputed:     0


#### primary_neighborhood & secondary_neighborhood

In [74]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.latitude.notna() &
        df_crime.longitude.notna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 8,446,456 :::::

--- Showing 5 of 8446456 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
356908,2023,Austin,AUSTIN,170037750.826,25,25,AUSTIN,199254203.427,28,011,4,1113,60644,1145364,1900573,41.883171,-87.741677
1614099,2018,Chicago Lawn,"MARQUETTE PARK,GAGE PARK",98279465.1151,66,66,CHICAGO LAWN,98279465.1151,16,008,1,0824,60629,1156734,1865013,41.785368,-87.700886
1769584,2017,Chatham,"CHATHAM,BURNSIDE",82320670.3112,44,44,CHATHAM,82320670.3112,21,006,2,0622,60620,1176740,1847273,41.73626,-87.628069
8132929,2001,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1651,60666,1100635,1934208,41.9762,-87.905312
2374743,2015,Lake View,LAKE VIEW,76561177.2562,06,6,LAKE VIEW,87214799.2728,47,019,3,1912,60613,1164934,1926849,41.954882,-87.669067


In [75]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['primary_neighborhood','secondary_neighborhood']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,446,456
Rows updated:       26
Rows not updated:   8,446,430
Values imputed:     52


#### ward

In [76]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.latitude.notna() &
        df_crime.longitude.notna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 8,446,456 :::::

--- Showing 5 of 8446456 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
356908,2023,Austin,AUSTIN,170037750.826,25,25,AUSTIN,199254203.427,28,011,4,1113,60644,1145364,1900573,41.883171,-87.741677
1614099,2018,Chicago Lawn,"MARQUETTE PARK,GAGE PARK",98279465.1151,66,66,CHICAGO LAWN,98279465.1151,16,008,1,0824,60629,1156734,1865013,41.785368,-87.700886
1769584,2017,Chatham,"CHATHAM,BURNSIDE",82320670.3112,44,44,CHATHAM,82320670.3112,21,006,2,0622,60620,1176740,1847273,41.73626,-87.628069
8132929,2001,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1651,60666,1100635,1934208,41.9762,-87.905312
2374743,2015,Lake View,LAKE VIEW,76561177.2562,06,6,LAKE VIEW,87214799.2728,47,019,3,1912,60613,1164934,1926849,41.954882,-87.669067


In [77]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude', 
        'district', 'beat']
update_cols = ['ward']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,446,456
Rows updated:       27
Rows not updated:   8,446,429


#### zip_code

In [78]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.latitude.notna() &
        df_crime.longitude.notna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 8,446,456 :::::

--- Showing 5 of 8446456 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
356908,2023,Austin,AUSTIN,170037750.826,25,25,AUSTIN,199254203.427,28,011,4,1113,60644,1145364,1900573,41.883171,-87.741677
1614099,2018,Chicago Lawn,"MARQUETTE PARK,GAGE PARK",98279465.1151,66,66,CHICAGO LAWN,98279465.1151,16,008,1,0824,60629,1156734,1865013,41.785368,-87.700886
1769584,2017,Chatham,"CHATHAM,BURNSIDE",82320670.3112,44,44,CHATHAM,82320670.3112,21,006,2,0622,60620,1176740,1847273,41.73626,-87.628069
8132929,2001,O'Hare,OHARE,371835607.687,76,76,OHARE,371835607.687,41,016,5,1651,60666,1100635,1934208,41.9762,-87.905312
2374743,2015,Lake View,LAKE VIEW,76561177.2562,06,6,LAKE VIEW,87214799.2728,47,019,3,1912,60613,1164934,1926849,41.954882,-87.669067


In [79]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,446,456
Rows updated:       26
Rows not updated:   8,446,430
Values imputed:     52


### Final Impute

#### longitude & latitude

In [80]:
# mask
mask = (
        df_crime.latitude.isna() & 
        df_crime.longitude.isna() &
        df_crime.beat.notna() 
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 96,633 :::::

--- Showing 5 of 96633 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,<NA>,<NA>,<NA>,<NA>
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,<NA>,<NA>,<NA>,<NA>
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,<NA>,<NA>,<NA>,<NA>
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,<NA>,<NA>,<NA>,<NA>
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,<NA>,<NA>,<NA>,<NA>


In [81]:
# Composite key
keys = ['beat']
update_cols = ['latitude', 'longitude']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 96,633
Rows updated:       96,633
Rows not updated:   0
Values imputed:     193,266


In [82]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,<NA>,<NA>,41.654917,-87.604963
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,<NA>,<NA>,42.019337,-87.680511
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,<NA>,<NA>,41.947048,-87.646931
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,<NA>,<NA>,41.900769,-87.643107
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,<NA>,<NA>,41.930482,-87.757325


#### x_coordinate & y_coordinate

In [83]:
# mask
mask = (
        df_crime.latitude.notna() & 
        df_crime.longitude.notna() &
        df_crime.x_coordinate.isna() &
        df_crime.y_coordinate.isna() 
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 96,633 :::::

--- Showing 5 of 96633 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,<NA>,<NA>,41.654917,-87.604963
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,<NA>,<NA>,42.019337,-87.680511
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,<NA>,<NA>,41.947048,-87.646931
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,<NA>,<NA>,41.900769,-87.643107
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,<NA>,<NA>,41.930482,-87.757325


In [84]:
# Composite key
keys = ['latitude', 'longitude']
update_cols = ['x_coordinate', 'y_coordinate']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 96,633
Rows updated:       96,633
Rows not updated:   0
Values imputed:     193,266


In [85]:
# Display
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
5945800,2005,<NA>,<NA>,<NA>,54,<NA>,<NA>,<NA>,09,005,2,0533,<NA>,1183297,1817685,41.654917,-87.604963
7497075,2002,<NA>,<NA>,<NA>,01,<NA>,<NA>,<NA>,49,024,3,2424,<NA>,1161642,1950313,42.019337,-87.680511
3898010,2010,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,46,019,3,1925,<NA>,1170978,1924042,41.947048,-87.646931
7476658,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,43,018,3,1822,<NA>,1172154,1907186,41.900769,-87.643107
1786379,2017,<NA>,<NA>,<NA>,19,<NA>,<NA>,<NA>,30,025,5,2514,<NA>,1140987,1917784,41.930482,-87.757325


#### community_code

In [86]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.community_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 508,067 :::::

--- Showing 5 of 508067 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8158285,2001,West Town,"WICKER PARK,WEST TOWN",58507728.4211,<NA>,24,WEST TOWN,127562904.597,08,012,3,1323,60642,1169878,1905589,41.896437,-87.651514
8244347,2001,Washington Heights,"WASHINGTON HEIGHTS,ROSELAND",79635752.8769,<NA>,73,WASHINGTON HEIGHTS,79635752.8769,08,022,2,2232,60643,1171532,1835736,41.704716,-87.647486
7935316,2001,Near South Side,NEAR SOUTH SIDE,34252582.7003,<NA>,33,NEAR SOUTH SIDE,49769639.4541,08,001,3,0132,60605,1177377,1894944,41.867059,-87.624294
8241524,2001,Bridgeport,BRIDGEPORT,58291519.276,<NA>,60,BRIDGEPORT,58291519.2767,08,009,1,0922,60609,1171536,1880297,41.826997,-87.646168
8234221,2001,Rogers Park,ROGERS PARK,51259902.4506,<NA>,1,ROGERS PARK,51259902.4506,08,024,3,2422,60626,1164934,1950380,42.019451,-87.668395


In [87]:
# Composite key
keys = ['x_coordinate', 'y_coordinate']
update_cols = ['community_code']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 508,067
Rows updated:       49,421
Rows not updated:   458,646


#### Ward

In [88]:
# update mask
mask = (df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.ward.isna()
       )

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 9,318 :::::

--- Showing 5 of 9318 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
7805263,2002,<NA>,<NA>,<NA>,29,<NA>,<NA>,<NA>,<NA>,010,4,1014,<NA>,1150382,1890933,41.856622,-87.723501
7772379,2002,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,<NA>,019,3,2332,<NA>,1172968,1921410,41.939782,-87.639695
7841747,2001,<NA>,<NA>,<NA>,39,<NA>,<NA>,<NA>,<NA>,002,1,2123,<NA>,1183840,1878113,41.820725,-87.601095
7746060,2002,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,<NA>,018,3,1824,<NA>,1175895,1908328,41.903819,-87.629332
8045887,2001,<NA>,<NA>,<NA>,43,<NA>,<NA>,<NA>,<NA>,003,1,0331,<NA>,1190752,1860397,41.771947,-87.576311


In [89]:
# Composite key
keys = ['x_coordinate', 'y_coordinate']
update_cols = ['ward']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 9,318
Rows updated:       9,290
Rows not updated:   28


#### primary_neighborhood

In [90]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.primary_neighborhood.isna()

)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 118,472 :::::

--- Showing 5 of 118472 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
1919447,2017,<NA>,<NA>,<NA>,72,<NA>,<NA>,<NA>,19,022,2,2213,<NA>,1168764,1839148,41.714139,-87.657524
726694,2022,<NA>,<NA>,<NA>,29,<NA>,<NA>,<NA>,28,011,4,1135,<NA>,1157839,1895835,41.869925,-87.695997
7907211,2001,<NA>,<NA>,<NA>,16,<NA>,<NA>,<NA>,08,017,5,1733,<NA>,1157107,1921352,41.93996,-87.69799
2404682,2015,<NA>,<NA>,<NA>,08,<NA>,<NA>,<NA>,42,018,3,1834,<NA>,1176928,1905370,41.895679,-87.625627
1206607,2020,<NA>,<NA>,<NA>,06,<NA>,<NA>,<NA>,44,019,3,1924,<NA>,1169211,1921469,41.940027,-87.653501


In [91]:
# Composite key
keys = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
update_cols = ['primary_neighborhood', 'neighborhood_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 118,472
Rows updated:       96,629
Rows not updated:   21,843
Values imputed:     193,258


In [92]:
# Display
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
1919447,2017,Washington Heights,<NA>,79635752.8769,72,<NA>,<NA>,<NA>,19,022,2,2213,<NA>,1168764,1839148,41.714139,-87.657524
726694,2022,Garfield Park,<NA>,89976069.5947,29,<NA>,<NA>,<NA>,28,011,4,1135,<NA>,1157839,1895835,41.869925,-87.695997
7907211,2001,Avondale,<NA>,55290595.482,16,<NA>,<NA>,<NA>,08,017,5,1733,<NA>,1157107,1921352,41.93996,-87.69799
2404682,2015,River North,<NA>,38766442.5194,08,<NA>,<NA>,<NA>,42,018,3,1834,<NA>,1176928,1905370,41.895679,-87.625627
1206607,2020,Lake View,<NA>,76561177.2562,06,<NA>,<NA>,<NA>,44,019,3,1924,<NA>,1169211,1921469,41.940027,-87.653501


#### secondary_neighborhood

In [93]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.secondary_neighborhood.isna()

)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 118,472 :::::

--- Showing 5 of 118472 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
1919447,2017,Washington Heights,<NA>,79635752.8769,72,<NA>,<NA>,<NA>,19,022,2,2213,<NA>,1168764,1839148,41.714139,-87.657524
726694,2022,Garfield Park,<NA>,89976069.5947,29,<NA>,<NA>,<NA>,28,011,4,1135,<NA>,1157839,1895835,41.869925,-87.695997
7907211,2001,Avondale,<NA>,55290595.482,16,<NA>,<NA>,<NA>,08,017,5,1733,<NA>,1157107,1921352,41.93996,-87.69799
2404682,2015,River North,<NA>,38766442.5194,08,<NA>,<NA>,<NA>,42,018,3,1834,<NA>,1176928,1905370,41.895679,-87.625627
1206607,2020,Lake View,<NA>,76561177.2562,06,<NA>,<NA>,<NA>,44,019,3,1924,<NA>,1169211,1921469,41.940027,-87.653501


In [94]:
# Composite key
keys = ['x_coordinate', 'y_coordinate']
update_cols = ['secondary_neighborhood']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 118,472
Rows updated:       96,629
Rows not updated:   21,843


#### ca_community_code

In [95]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.ca_community_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 96,707 :::::

--- Showing 5 of 96707 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
2271024,2016,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,19,<NA>,<NA>,<NA>,30,025,5,2515,<NA>,1139984,1916399,41.9267,-87.761045
6071169,2005,South Chicago,SOUTH CHICAGO,93272185.0092,48,<NA>,<NA>,<NA>,10,004,2,0423,<NA>,1196664,1849421,41.741683,-87.555004
7278931,2003,Lake View,LAKE VIEW,76561177.2562,06,<NA>,<NA>,<NA>,44,019,3,1924,<NA>,1169211,1921469,41.940027,-87.653501
5934781,2005,Lincoln Square,LINCOLN SQUARE,71352328.2399,03,<NA>,<NA>,<NA>,47,020,3,2032,<NA>,1161202,1931897,41.968812,-87.682645
938554,2021,Garfield Park,GARFIELD PARK,89976069.5947,27,<NA>,<NA>,<NA>,28,011,4,1134,<NA>,1154092,1896208,41.871024,-87.709743


In [96]:
# Composite key
keys = ['x_coordinate', 'y_coordinate']
update_cols = ['ca_community_code', 'ca_community_name', 'ca_community_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 96,707
Rows updated:       96,633
Rows not updated:   74
Values imputed:     289,899


#### zip_code

In [97]:
# mask
mask = (
        df_crime.x_coordinate.notna() & 
        df_crime.y_coordinate.notna() &
        df_crime.zip_code.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 118,413 :::::

--- Showing 5 of 118413 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
933648,2021,River North,RIVER NORTH,38766442.5194,08,8,NEAR NORTH SIDE,76675895.9728,42,018,3,1831,<NA>,1176114,1903564,41.890742,-87.628671
3488056,2012,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,36,025,5,2513,<NA>,1127853,1909983,41.909307,-87.805767
5473966,2006,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,36,025,5,2513,<NA>,1127853,1909983,41.909307,-87.805767
787494,2022,"Little Italy, UIC","LITTLE ITALY, UIC",71376244.1225,28,28,NEAR WEST SIDE,158492466.554,27,012,3,1231,<NA>,1166334,1893853,41.864309,-87.664866
7528651,2002,West Town,"WICKER PARK,WEST TOWN",58507728.4211,24,24,WEST TOWN,127562904.597,01,012,3,1322,<NA>,1165614,1905718,41.896883,-87.667171


In [98]:
# Composite key
keys = ['x_coordinate', 'y_coordinate']
update_cols = ['zip_code', 'zip_code_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 118,413
Rows updated:       96,629
Rows not updated:   21,784
Values imputed:     193,258


### Last Clean-Up: Impute

In [99]:
# nans
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                         Count Percentage
community_code          458646    5.3686%
neighborhood_area        21869    0.2560%
primary_neighborhood     21843    0.2557%
secondary_neighborhood   21843    0.2557%
zip_code                 21784    0.2550%
zip_code_area            21784    0.2550%
primary_description      19371    0.2267%
secondary_description    19371    0.2267%
index_code               19371    0.2267%
location_description     15968    0.1869%
district_location          328    0.0038%
ca_community_code           74    0.0009%
ca_community_name           74    0.0009%
ca_community_area           74    0.0009%
ward                        28    0.0003%


In [100]:
# mask
mask = (
        df_crime.latitude.notna() & 
        df_crime.longitude.notna() &
        df_crime.x_coordinate.isna() &
        df_crime.y_coordinate.isna() 
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 0 :::::

--- Showing 0 of 0 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [101]:
# Composite key
keys = ['latitude', 'longitude']
update_cols = ['x_coordinate', 'y_coordinate']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0
Values imputed:     0


In [102]:
# Display
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


#### community_code & ward

In [103]:
# mask
mask = (
    df_crime.community_code.isna() |
    df_crime.ward.isna()
)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 458,674 :::::

--- Showing 5 of 458674 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8170640,2001,Hermosa,"BELMONT CRAIGIN,HERMOSA",32602059.4055,<NA>,20,HERMOSA,32602059.4055,08,025,5,2534,60639,1147366,1912803,41.916694,-87.734011
8221334,2001,Fuller Park,BACK OF THE YARDS,19916704.8692,<NA>,37,FULLER PARK,19916704.8692,08,009,1,0934,60609,1175241,1869234,41.796557,-87.632905
8248103,2001,West Ridge,WEST RIDGE,98429094.8621,<NA>,2,WEST RIDGE,98429094.8621,08,024,3,2411,60645,1158779,1946100,42.007836,-87.691163
7873572,2001,Riverdale,RIVERDALE,98389497.4143,<NA>,54,RIVERDALE,98389497.4143,08,005,2,0533,60827,1183974,1817992,41.655743,-87.602476
7994142,2001,Englewood,ENGLEWOOD,173600015.009,<NA>,67,WEST ENGLEWOOD,87947691.9478,08,007,1,0725,60636,1166793,1862823,41.779149,-87.664068


In [104]:
# Composite key
keys = ['year', 'district', 'beat']
update_cols = ['community_code', 'ward']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 458,674
Rows updated:       458,669
Rows not updated:   5
Values imputed:     458,669


In [105]:
# Verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
8170640,2001,Hermosa,"BELMONT CRAIGIN,HERMOSA",32602059.4055,23,20,HERMOSA,32602059.4055,08,025,5,2534,60639,1147366,1912803,41.916694,-87.734011
8221334,2001,Fuller Park,BACK OF THE YARDS,19916704.8692,61,37,FULLER PARK,19916704.8692,08,009,1,0934,60609,1175241,1869234,41.796557,-87.632905
8248103,2001,West Ridge,WEST RIDGE,98429094.8621,02,2,WEST RIDGE,98429094.8621,08,024,3,2411,60645,1158779,1946100,42.007836,-87.691163
7873572,2001,Riverdale,RIVERDALE,98389497.4143,54,54,RIVERDALE,98389497.4143,08,005,2,0533,60827,1183974,1817992,41.655743,-87.602476
7994142,2001,Englewood,ENGLEWOOD,173600015.009,67,67,WEST ENGLEWOOD,87947691.9478,08,007,1,0725,60636,1166793,1862823,41.779149,-87.664068


#### Police District & Sector & beat

In [106]:
# mask
mask = (
        df_crime.year.notna() &
        (df_crime.sector.isna() |    
        df_crime.district.isna())

)

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 0 :::::

--- Showing 0 of 0 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [107]:
# Composite key
keys = ['year', 'beat', 'community_code', 'ward', 'zip_code']
update_cols = ['sector', 'district']
# Update
df_crime1 = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0
Values imputed:     0


#### primary_neighborhood

In [108]:
# mask
mask = (
        df_crime.beat.notna() &
    (
        df_crime.primary_neighborhood.isna() |
        df_crime.secondary_neighborhood.isna() |
        df_crime.neighborhood_area.isna()
    )
       )

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 21,869 :::::

--- Showing 5 of 21869 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
2905549,2013,<NA>,<NA>,<NA>,01,76,OHARE,371835607.687,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049
5042318,2008,<NA>,<NA>,<NA>,74,76,OHARE,371835607.687,19,022,2,2211,<NA>,1151290,1828011,41.683936,-87.721811
1095332,2020,<NA>,<NA>,<NA>,25,76,OHARE,371835607.687,29,015,4,1513,<NA>,1136382,1899655,41.880817,-87.774681
7771731,2002,<NA>,<NA>,<NA>,69,76,OHARE,371835607.687,08,024,3,2424,<NA>,1161566,1950354,42.019451,-87.68079
5480765,2006,<NA>,<NA>,<NA>,01,76,OHARE,371835607.687,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049


In [109]:
# Composite key
keys = ['year', 'beat']
update_cols = ['primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 21,869
Rows updated:       21,864
Rows not updated:   5
Values imputed:     65,540


In [110]:
# verify
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
2905549,2013,Rogers Park,ROGERS PARK,51259902.4506,01,76,OHARE,371835607.687,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049
5042318,2008,Mount Greenwood,"MOUNT GREENWOOD,MORGAN PARK",75584290.0209,74,76,OHARE,371835607.687,19,022,2,2211,<NA>,1151290,1828011,41.683936,-87.721811
1095332,2020,Austin,AUSTIN,170037750.826,25,76,OHARE,371835607.687,29,015,4,1513,<NA>,1136382,1899655,41.880817,-87.774681
7771731,2002,Rogers Park,ROGERS PARK,51259902.4506,69,76,OHARE,371835607.687,08,024,3,2424,<NA>,1161566,1950354,42.019451,-87.68079
5480765,2006,Rogers Park,ROGERS PARK,51259902.4506,01,76,OHARE,371835607.687,49,024,3,2422,<NA>,1163126,1950347,42.019399,-87.675049


#### Update Rest NaNs

In [111]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                        Count Percentage
zip_code                21784    0.2550%
zip_code_area           21784    0.2550%
primary_description     19371    0.2267%
secondary_description   19371    0.2267%
index_code              19371    0.2267%
location_description    15968    0.1869%
district_location         328    0.0038%
ca_community_code          74    0.0009%
ca_community_name          74    0.0009%
ca_community_area          74    0.0009%
community_code              5    0.0001%
primary_neighborhood        5    0.0001%
secondary_neighborhood      5    0.0001%
neighborhood_area           5    0.0001%


In [112]:
# mask
mask = (
    df_crime.year.notna() &
    df_crime.beat.notna()
       )

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 8,543,089 :::::

--- Showing 5 of 8543089 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude
6005870,2005,South Chicago,SOUTH CHICAGO,93272185.0092,46,46,SOUTH CHICAGO,93272185.0092,10,004,2,0424,60617,1198076,1846205,41.732823,-87.549938
2904737,2013,Burnside,"CHATHAM,BURNSIDE",16995983.2737,47,47,BURNSIDE,16995983.2737,08,004,2,0413,60619,1184148,1844756,41.729183,-87.601007
5835722,2006,New City,BACK OF THE YARDS,134636963.254,61,61,NEW CITY,134636963.254,16,009,1,0932,60609,1166927,1868751,41.795414,-87.663407
3503885,2011,Riverdale,RIVERDALE,98389497.4143,54,54,RIVERDALE,98389497.4143,09,005,2,0533,60827,1185714,1818799,41.657917,-87.596084
4490856,2009,Humboldt Park,HUMBOLDT PARK,125010425.593,24,24,WEST TOWN,127562904.597,01,014,5,1421,60647,1157452,1910783,41.910951,-87.69701


In [113]:
# Composite key
keys = ['year', 'beat']
update_cols = ['sector', 'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area', 
               'zip_code', 'zip_code_area', 'district', 'community_code',
               'ca_community_code', 'ca_community_name', 'ca_community_area']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 8,543,089
Rows updated:       21,858
Rows not updated:   8,521,231
Values imputed:     43,785


### DataFrame Maint

In [114]:
# collapse fragmented blocks
df_crime = df_crime.copy()
# display
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_code,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,sector,ca_community_code,ca_community_name,ca_community_area,latitude,longitude,x_coordinate,y_coordinate,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location
8265318,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSSESS - CANNABIS 30 GRAMS OR LESS,N,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,08,43,2001,2015-08-17 15:03:40,18,60649,80526075.8505,South Shore,"SOUTH SHORE, GRAND CROSSING",81812716.3904,1,43,SOUTH SHORE,81812716.3958,41.764219,-87.582549,1189075,1857566,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False,Grand Crossing
7080787,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,I,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,08,69,2003,2015-08-17 15:03:40,06,60619,167872012.337,Grand Crossing,"SOUTH SHORE, GRAND CROSSING",98853167.7093,2,69,GREATER GRAND CROSSING,98853167.7093,41.755291,-87.59816,1184844,1854276,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True,Gresham
6396203,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSSESS - HEROIN (WHITE),N,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,27,23,2004,2018-02-28 15:56:25,18,60624,99418122.6738,Humboldt Park,HUMBOLDT PARK,125010425.593,4,23,HUMBOLDT PARK,100480876.502,41.892451,-87.719888,1151273,1903996,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False,Harrison
5818733,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,I,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,29,19,2006,2018-02-28 15:56:25,05,60707,48519709.6539,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,5,19,BELMONT CRAGIN,109099414.689,41.928096,-87.78561,1133296,1916864,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True,Grand Central
5312174,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSSESS - CANNABIS MORE THAN 30 GRAMS,N,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,31,19,2007,2018-02-28 15:56:25,18,60639,127476051.26,Belmont Cragin,"BELMONT CRAGIN,HERMOSA",109099407.211,5,19,BELMONT CRAGIN,109099414.689,41.921066,-87.747452,1143697,1914371,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False,Grand Central


#### Examine Each NaNs

In [115]:
# NaNs
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                        Count Percentage
primary_description     19371    0.2267%
secondary_description   19371    0.2267%
index_code              19371    0.2267%
location_description    15968    0.1869%
district_location         328    0.0038%
zip_code                    5    0.0001%
zip_code_area               5    0.0001%
primary_neighborhood        5    0.0001%
secondary_neighborhood      5    0.0001%
neighborhood_area           5    0.0001%


#### Check x_coordinate	& y_coordinate & latitude & longitude

In [116]:
# update mask
mask = (df_crime.x_coordinate.isna() &
        df_crime.longitude.notna() &
        df_crime.district.notnull()
       )
print(f"NaN count: {mask.sum()}")
df_crime.loc[mask, cols].head()

NaN count: 0


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


#### sector

In [117]:
# update mask
mask = (
        df_crime.sector.isna()
       )

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 0 :::::

--- Showing 0 of 0 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [118]:
# Composite key
keys = ['district', 'beat']
update_cols = ['sector']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0


In [119]:
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [120]:
# update mask
mask = (
        df_crime.sector.isna()
       )

# Display
idx = get_sample_report(data_df=df_crime, bool_mask=mask)
df_crime.loc[idx, cols]

::::: Missing: 0 :::::

--- Showing 0 of 0 eligible rows ---


,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [121]:
# Composite key
keys = ['district']
update_cols = ['sector']
# Update
df_crime = geo.impute_data(data_df=df_crime, keys=keys, update_cols=update_cols, mask=mask)

----- Imputation complete -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0


In [122]:
df_crime.loc[idx, cols]

,year,primary_neighborhood,secondary_neighborhood,neighborhood_area,community_code,ca_community_code,ca_community_name,ca_community_area,ward,district,sector,beat,zip_code,x_coordinate,y_coordinate,latitude,longitude


In [123]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,543,089) ---
                        Count Percentage
primary_description     19371    0.2267%
secondary_description   19371    0.2267%
index_code              19371    0.2267%
location_description    15968    0.1869%
district_location         328    0.0038%
zip_code                    5    0.0001%
zip_code_area               5    0.0001%
primary_neighborhood        5    0.0001%
secondary_neighborhood      5    0.0001%
neighborhood_area           5    0.0001%


In [124]:
# Final State
print(f"Shape: {df_crime.shape}")
print(f"\nDtypes:\n{df_crime.dtypes}")
utils.any_nans(df_crime)

Shape: (8543089, 41)

Dtypes:
case_number                                                 string[pyarrow]
date                                                  timestamp[s][pyarrow]
block                                                       string[pyarrow]
iucr                                                        string[pyarrow]
primary_description                                         string[pyarrow]
secondary_description                                       string[pyarrow]
index_code                                                  string[pyarrow]
primary_type                                                string[pyarrow]
description                                                 string[pyarrow]
location_description                                        string[pyarrow]
arrest                                                                 bool
domestic                                                               bool
beat                                                      

* Decision: retain rows - nulls preserved for auditability and future validation.
* Downstream analysis does not depend on community_code or ward features.

## Total Time

In [125]:
# total elapsed time
elapsed = time.time() - start
print(f"{elapsed:.2f}s to process {df_crime.shape[0]:,} rows")

108.35s to process 8,543,089 rows


## Save using PyArrow

In [126]:
# Save as Arrow
feather.write_feather(df_crime, "../Data/crime_data.feather")